In [1]:
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [3]:
train_data = pd.read_csv("samsum-train.csv")
validation_data = pd.read_csv("samsum-validation.csv")

In [6]:
train_data = train_data.sample(n=10000,random_state=42).reset_index(drop=True)
validation_data = validation_data.sample(n=700,random_state=42).reset_index(drop=True)

In [7]:
train_data.shape

(10000, 3)

# Data  pre-processing

In [8]:
def clean_data(text):
  text = re.sub(r"\r\n"," ",text) # lines
  text = re.sub(r"\s+"," ",text) # speac
  text = re.sub(r"<.*?>"," ",text) # html tags
  text = text.strip().lower()

  return text

In [9]:
train_data['dialogue'] = train_data['dialogue'].fillna('').apply(clean_data)
train_data['summary'] = train_data['summary'].fillna('').apply(clean_data)


validation_data['dialogue'] = validation_data['dialogue'].fillna('').apply(clean_data)
validation_data['summary'] = validation_data['summary'].fillna('').apply(clean_data)

In [10]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [11]:
def tokeniz(data):
  inputs = tokenizer(data['dialogue'],padding='max_length',max_length=512, truncation=True)
  target = tokenizer(data['summary'],padding='max_length',max_length=200, truncation=True)

  inputs['labels'] = target['input_ids']
  return inputs

train_dataset = train_data.apply(tokeniz,axis=1).tolist()
validation_dataset = validation_data.apply(tokeniz,axis=1).tolist()

In [12]:
train_dataset[0]

{'input_ids': [12, 635, 63, 10, 3, 23, 31, 162, 7122, 39, 2412, 367, 230, 6, 34, 31, 7, 59, 1480, 2576, 83, 10, 2049, 55, 3, 29, 15, 173, 10, 207, 1506, 6, 44, 709, 8, 2412, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

# Let's build the model

In [13]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [14]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [15]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs=8,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy ='epoch',
    warmup_steps = 500
)

In [16]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset
)

# Train the model

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.308708,0.261451
2,0.285626,0.252372
3,0.275327,0.247754
4,0.264266,0.245779
5,0.263073,0.244061
6,0.256236,0.243560
7,0.252676,0.243237
8,0.251521,0.243209


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10000, training_loss=0.4315227821350098, metrics={'train_runtime': 4397.7925, 'train_samples_per_second': 18.191, 'train_steps_per_second': 2.274, 'total_flos': 1.082734411776e+16, 'train_loss': 0.4315227821350098, 'epoch': 8.0})

In [18]:
model.save_pretrained("./save_summary_model")
tokenizer.save_pretrained('./save_summary_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./save_summary_model/tokenizer_config.json',
 './save_summary_model/tokenizer.json')

In [19]:
model = T5ForConditionalGeneration.from_pretrained("./save_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./save_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [20]:
def summary(dialogue):
  dialogue=clean_data(dialogue)

  inputs = tokenizer(
       dialogue,
       padding='max_length',
       max_length=512,
       truncation=True,
       return_tensors = "pt"
   )

  targets = model.generate(
       input_ids = inputs["input_ids"],
       attention_mask = inputs['attention_mask'],
       max_length =200,
       num_beams = 5,
       early_stopping=True
   )

  text_summary = tokenizer.decode(targets[0],skip_special_tokens=True)

  return text_summary

In [23]:
test_dialogue = """Virat Kohli is one of the most well-known and successful cricketers of modern times. He was born on November 5, 1988, in Delhi, India. From a young age, Kohli showed a strong passion for cricket and worked extremely hard to develop his skills. His dedication and determination helped him become one of the leading batsmen in international cricket.

Kohli rose to international fame when he led the Indian Under-19 cricket team to victory in the 2008 Under-19 Cricket World Cup. Soon after, he made his debut for the Indian senior cricket team. He became famous for his aggressive batting style, excellent fitness, consistency, and strong leadership qualities.

As a batsman, Kohli has achieved numerous records across different formats of cricket. He has been particularly successful in One Day International cricket, where he has scored a large number of runs and centuries. His ability to chase difficult targets has earned him a reputation as one of the finest run chasers in cricket.

Kohli also served as captain of the Indian cricket team. Under his leadership, India achieved several important victories, particularly in Test cricket. His emphasis on fitness and discipline also influenced the team's overall approach to professional cricket.

Apart from his cricketing skills, Kohli is known for his competitive personality and commitment to fitness. He has inspired many young athletes to focus on discipline, hard work, and healthy lifestyles.

Virat Kohli has received numerous awards and honors for his contribution to cricket. His journey from a young cricket enthusiast in Delhi to an international cricket superstar is an inspiring example of dedication and perseverance.

In conclusion, Virat Kohli is not only a talented cricketer but also an inspiration to millions of people around the world. His passion, discipline, and determination have made him an important figure in modern cricket."""

In [22]:
summarizer = summary(test_dialogue)

print(f"Summary : {summarizer}")

Summary : virat kohli is one of the most well-known and successful cricketers of modern times. he was born in delhi, india, on november 5, 1988. kohli rose to international fame when he led the indian under-19 cricket team to victory in the 2008 under-19 cricket world cup.
